# 🔬 Notebook 3: S3 — Deep Dives

Five short, runnable deep dives:

1. **Presigned URLs** — grant one-shot access without leaking keys.
2. **Consistent hashing** — pick a storage node without reshuffling
   the world when you add capacity.
3. **Sharding the metadata plane** — hash vs range partitioning, and why the
   choice decides whether `LIST` costs one shard or all of them.
4. **Erasure coding** — intuition with XOR parity.
5. **Lifecycle policies** — move cold data to cheap storage (or
   delete it) automatically.

Each dive follows the same shape: show the obvious **bad** approach,
then the **better** one with code.

## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Then in VS Code pick the `.venv` kernel from the top-right of the notebook. If
it doesn't show up: `Cmd+Shift+P` → **Reload Window** and try again.

Everything in this lab is **pure Python** — no databases, no Docker. You can
run it on a laptop in a few seconds.


## 1️⃣ Presigned URLs

### Bad: share the master credentials 😱

Imagine a mobile app that needs to download `photos/cat.jpg`. The
quickest thing to ship: put the AWS access key in the app. This is
a catastrophe:

- No expiry — leak = forever access.
- No scope — leak = full account access.
- Impossible to rotate without shipping a new app version.


### Better: presigned URL with HMAC + expiry 🔐

Idea: the server knows a secret, the client doesn't. The server
computes `sig = HMAC(secret, method|bucket|key|expiry)` and hands the
client a URL that includes the signature and the expiry. The server
re-computes the signature on arrival and compares.

Any tampering (different method, different object, expired clock,
changed signature) makes the re-computed HMAC mismatch, so access is
refused.


In [ ]:
import hmac, hashlib, time, urllib.parse

SECRET_BYTES = b"super-secret"   # lives ONLY on the server

def presign(method: str, bucket: str, key: str, ttl: int = 60) -> str:
    expires = int(time.time()) + ttl
    msg = f"{method}|{bucket}|{key}|{expires}".encode()
    sig = hmac.new(SECRET_BYTES, msg, hashlib.sha256).hexdigest()
    return f"/{bucket}/{key}?exp={expires}&sig={sig}"

def verify(method: str, bucket: str, key: str, url: str):
    try:
        qs = urllib.parse.parse_qs(url.split("?", 1)[1])
        exp = int(qs["exp"][0])
        sig = qs["sig"][0]
    except (IndexError, KeyError, ValueError):
        return False, "malformed"
    if time.time() > exp:
        return False, "expired"
    msg = f"{method}|{bucket}|{key}|{exp}".encode()
    expected = hmac.new(SECRET_BYTES, msg, hashlib.sha256).hexdigest()
    # compare_digest is constant-time -> no timing side-channels
    ok = hmac.compare_digest(sig, expected)
    return ok, "ok" if ok else "bad sig"

url = presign("GET", "photos", "cat.jpg", ttl=60)
print("URL:", url)
print("valid GET:      ", verify("GET", "photos", "cat.jpg", url))
print("wrong method:   ", verify("PUT", "photos", "cat.jpg", url))
print("wrong object:   ", verify("GET", "photos", "dog.jpg", url))
print("tampered sig:   ", verify("GET", "photos", "cat.jpg", url[:-1] + "0"))

short = presign("GET", "photos", "cat.jpg", ttl=0)
time.sleep(1)
print("expired:        ", verify("GET", "photos", "cat.jpg", short))


Why this design is nice:

- **No server round-trip for the download itself** — the client hits
  the storage URL directly, saving bandwidth on the API front-end.
- **Scoped** — a leak only buys you *that one object*.
- **Time-bounded** — even a leak is fine after the TTL.
- **Stateless** — the server doesn't remember the URL; it just
  re-derives the signature and compares.


## 2️⃣ Consistent hashing for shard placement

### Bad: `hash(key) % N` 💥

Pick a node for each key with `hash(key) % N`. Simple. Now add one
more node. Suddenly **most** keys map to a different node, so we
reshuffle the entire dataset just to grow by 25%.


In [ ]:
def mod_placement(key, nodes):
    return nodes[hash(key) % len(nodes)]

keys = [f"object-{i}" for i in range(1000)]
before = [mod_placement(k, ["n1", "n2", "n3", "n4"])       for k in keys]
after  = [mod_placement(k, ["n1", "n2", "n3", "n4", "n5"]) for k in keys]

moved = sum(1 for b, a in zip(before, after) if b != a)
print(f"adding 1 node reshuffles {moved}/{len(keys)} keys ({moved/len(keys):.0%})")


### Better: consistent hashing with virtual nodes 💍

Imagine a ring of hash values. Each storage node hashes to several
points on the ring ("virtual nodes" or *vnodes*). A key hashes to
one point on the ring, and we walk clockwise until we hit a node.

Adding a node only takes over the slices closest to its vnodes — we
move about `1/N` of the keys. Vnodes smooth the distribution so no
one node ends up with a giant arc.


In [ ]:
from bisect import bisect_right
import hashlib

class ConsistentHashRing:
    def __init__(self, vnodes_per_node: int = 100):
        self.vnodes_per_node = vnodes_per_node
        self.ring: list = []            # sorted list of (hash, node)
        self._hashes: list[int] = []    # parallel key array, kept in sync

    @staticmethod
    def _h(s: str) -> int:
        return int(hashlib.md5(s.encode()).hexdigest(), 16)

    def _reindex(self):
        self.ring.sort()
        # Keep the bisect key array materialised. Rebuilding it inside
        # node_for() would make every lookup O(N) and quietly turn the whole
        # "O(log N) ring" story into a lie.
        self._hashes = [h for h, _ in self.ring]

    def add(self, node: str):
        self.ring.extend((self._h(f"{node}#{i}"), node) for i in range(self.vnodes_per_node))
        self._reindex()

    def remove(self, node: str):
        self.ring = [(h, n) for h, n in self.ring if n != node]
        self._reindex()

    def node_for(self, key: str) -> str:
        if not self.ring:
            raise RuntimeError("empty ring")
        idx = bisect_right(self._hashes, self._h(key)) % len(self.ring)
        return self.ring[idx][1]

    def nodes_for(self, key: str, count: int) -> list[str]:
        """Walk clockwise collecting `count` DISTINCT nodes — one per shard or
        replica. Skipping repeats is what stops two shards of the same object
        landing on the same machine."""
        if count > len({n for _, n in self.ring}):
            raise ValueError("not enough distinct nodes")
        start = bisect_right(self._hashes, self._h(key)) % len(self.ring)
        out: list[str] = []
        for step in range(len(self.ring)):
            node = self.ring[(start + step) % len(self.ring)][1]
            if node not in out:
                out.append(node)
                if len(out) == count:
                    return out
        return out


keys = [f"object-{i}" for i in range(1000)]
ring = ConsistentHashRing()
for n in ["n1", "n2", "n3", "n4"]:
    ring.add(n)

before = [ring.node_for(k) for k in keys]
ring.add("n5")
after  = [ring.node_for(k) for k in keys]

moved = sum(1 for b, a in zip(before, after) if b != a)
print(f"adding 1 node reshuffles {moved}/{len(keys)} keys ({moved/len(keys):.0%})")
print(f"~1/N of keys moved — not ~all of them.  (1/5 = 20%)")

# Placement for a 3-way spread must never double up on one node.
placement = ring.nodes_for("object-42", 3)
assert len(set(placement)) == 3, placement
print("3 distinct nodes for object-42:", placement)

# Honest cost: with few vnodes the ring is lumpy, and one node gets a giant arc.
from collections import Counter
for v in (1, 10, 100):
    r = ConsistentHashRing(vnodes_per_node=v)
    for n in ["n1", "n2", "n3", "n4"]:
        r.add(n)
    load = Counter(r.node_for(k) for k in keys)
    spread = max(load.values()) / min(load.values())
    print(f"vnodes={v:>3}: load per node {sorted(load.values())}  max/min = {spread:.2f}x")
print("→ vnodes buy balance and cost memory (ring entries) + slower add/remove.")

Real systems go further: they pick **R different nodes** from the
ring for each key (one per replica or shard) and make sure they're in
different failure domains. That's how erasure-coded shards get
spread across racks/zones without a central coordinator.


## 3️⃣ Sharding the metadata plane — and what it does to `LIST`

Notebook 1 sized the metadata plane at **365 billion rows**. That is many
databases, so the interesting question is how you split it. There are exactly
two answers and they are opposites.

- **Hash partitioning** — `shard = hash(key) % S`. Perfectly uniform load, and
  keys that look alike land nowhere near each other.
- **Range partitioning** — sorted key space cut into contiguous slices. Keys
  that look alike land together, and load follows whatever the key distribution
  happens to be.

`GET(key)` does not care: both send you to exactly one shard. `LIST(prefix)`
cares enormously, because a prefix is a *range*.

In [ ]:
import hashlib, bisect, random
from collections import Counter

SHARDS = 16
random.seed(1)

# A realistic key space: date-partitioned logs plus per-user photos.
keys: list[str] = []
for day in range(1, 61):
    for host in range(200):
        keys.append(f"logs/2024-06-{day:02d}/host-{host:03d}/app.log")
for user in range(2_000):
    for i in range(20):
        keys.append(f"photos/user-{user:05d}/img-{i:03d}.jpg")
print(f"{len(keys):,} keys across {SHARDS} metadata shards\n")

def hash_shard(key: str) -> int:
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % SHARDS

class RangeShardMap:
    """Contiguous slices of the sorted key space, split for equal row counts."""
    def __init__(self, all_keys, shards=SHARDS):
        s = sorted(all_keys)
        step = len(s) // shards
        self.splits = [s[i * step] for i in range(1, shards)]
    def shard(self, key: str) -> int:
        return bisect.bisect_right(self.splits, key)
    def shards_for_prefix(self, prefix: str) -> list[int]:
        upper = prefix[:-1] + chr(ord(prefix[-1]) + 1)
        return list(range(self.shard(prefix), self.shard(upper) + 1))

rng = RangeShardMap(keys)

# ---------- LIST: how many shards must we ask? ---------------------------
print(f"{'LIST prefix':<34}{'hash-sharded':>14}{'range-sharded':>15}")
for prefix in ["photos/user-00042/", "logs/2024-06-12/", "photos/", "logs/"]:
    hash_touched = SHARDS                       # hashing destroys locality: fan out to all
    rng_touched = len(rng.shards_for_prefix(prefix))
    plural = lambda n: f"{n} shard" + ("" if n == 1 else "s")
    print(f"{prefix:<34}{plural(hash_touched):>16}{plural(rng_touched):>15}")
print()
print("A LIST under hash partitioning is a scatter-gather over EVERY shard, and")
print("then a merge to restore key order for pagination. Its cost is O(shards),")
print("not O(results) — listing 3 objects costs the same as listing 3 million.")
print()
print("Range partitioning is only cheap for a NARROW prefix. 'photos/' spans most")
print("of the key space, so it fans out too — which is exactly right: that LIST")
print("really does have to read most of the data. The win is that cost now scales")
print("with the size of the ANSWER instead of the size of the cluster.")

In [ ]:
# ---------- …so range partitioning wins? Look at the write load. ----------
# Today's logs all share the prefix "logs/2024-06-60/", which is one contiguous
# slice of the key space — i.e. one shard.
todays_writes = [f"logs/2024-06-60/host-{h:03d}/app.log" for h in range(200)]

hash_load  = Counter(hash_shard(k)     for k in todays_writes)
range_load = Counter(rng.shard(k)      for k in todays_writes)

def describe(name, load):
    used = len(load)
    hottest = max(load.values())
    print(f"{name:<22} shards touched: {used:>2}/{SHARDS}   hottest shard takes "
          f"{hottest/len(todays_writes):>5.0%} of the writes")

describe("hash partitioning", hash_load)
describe("range partitioning", range_load)
print()
print("❌ Range partitioning concentrates every write for a monotonically")
print("   increasing key prefix (dates, timestamps, auto-increment ids) onto the")
print("   ONE shard that owns the tail of the key space. This is the reason S3's")
print("   old performance guidance told you to put a random prefix on your keys.")
print()

# ---------- The hashed-prefix workaround, and what it costs --------------
def hashed_prefix(key: str) -> str:
    return f"{hashlib.md5(key.encode()).hexdigest()[:2]}/{key}"

salted = [hashed_prefix(k) for k in todays_writes]
salted_map = RangeShardMap([hashed_prefix(k) for k in keys])
describe("range + hash prefix", Counter(salted_map.shard(k) for k in salted))
print("✅ writes are spread again…")
print("❌ …but 'logs/2024-06-12/' is no longer a contiguous range, so LIST is")
print("   back to scanning all", SHARDS, "shards. You cannot have both.")

### How real object stores resolve this

They range-partition, and then attack the hot-tail problem directly:

- **Automatic split and rebalance.** When a shard exceeds a size or request-rate
  threshold it splits in two and half moves elsewhere. S3 does this per key
  prefix (this is why the "use random prefixes" guidance was retired in 2018 —
  the system now adapts within minutes), and DynamoDB does the same with
  adaptive capacity. It does not eliminate the hotspot; it chases it, so a
  sudden burst still hurts until the split completes.
- **A secondary, asynchronously-maintained index for `LIST`** — which is exactly
  the eventual-consistency split you measured in Notebook 2. `GET` reads the
  authoritative range-partitioned store; `LIST` reads the index.
- **Keep listing out of the hot path.** The genuinely correct answer to "my job
  polls LIST every minute" is to stop polling: emit an event on write and drive
  the consumer from a queue.

**The trade-off in one line:** hash partitioning gives you uniform load and makes
`LIST` a scatter-gather; range partitioning gives you cheap `LIST` and makes hot
prefixes your operational problem. Object stores pick range plus a lot of
machinery, because `LIST` and pagination are part of the API contract and load
imbalance is merely expensive.

## 4️⃣ Erasure coding — intuition with XOR

Erasure coding = clever math that lets any `k` of `k+m` shards
rebuild the object. Real systems use **Reed–Solomon** over a finite
field, which is heavy math. But `k=2, m=1` has an intuitive form:
**parity = XOR of the data shards**. If one of the three is lost we
can recover it by XOR-ing the other two.

*(Why?* `a ⊕ b = p` ⇒ `a ⊕ p = b` ⇒ `b ⊕ p = a`. Try it!)

In [ ]:
def xor(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

data = b"hello world!!!!"
if len(data) % 2:
    data += b"\x00"
half = len(data) // 2
d1, d2 = data[:half], data[half:]
p = xor(d1, d2)

print("d1:", d1)
print("d2:", d2)
print("p :", p)

# Lose d2 -> rebuild from d1 XOR p
recovered_d2 = xor(d1, p)
print("recovered d2 == d2?", recovered_d2 == d2)
print("full:", d1 + recovered_d2)

# Lose d1 -> rebuild from d2 XOR p
recovered_d1 = xor(d2, p)
print("recovered d1 == d1?", recovered_d1 == d1)


To go beyond a single failure you need real Reed–Solomon (any `m`
failures tolerated), but the mental model is the same. In practice
you use a library — the cost per GB to encode/decode is tiny.


## 5️⃣ Lifecycle policies

Most objects are hot for a few days, then almost nobody touches them.
S3-style **storage classes** (Standard → Infrequent Access → Glacier)
trade latency for dollars. A **lifecycle policy** is a rule like:

> *Move to cold storage after 30 days; delete after 1 year.*

Here's the whole engine in ~15 lines.

In [ ]:
from datetime import datetime, timedelta, timezone

class LifecyclePolicy:
    def __init__(self, to_cold_after_days: int, expire_after_days: int):
        self.to_cold = timedelta(days=to_cold_after_days)
        self.expire  = timedelta(days=expire_after_days)

    def action(self, created_at: datetime, now: datetime | None = None) -> str:
        now = now or datetime.now(timezone.utc)
        age = now - created_at
        if age >= self.expire:
            return "delete"
        if age >= self.to_cold:
            return "move_to_cold"
        return "keep_hot"

p = LifecyclePolicy(to_cold_after_days=30, expire_after_days=365)
now = datetime.now(timezone.utc)
for days in [1, 31, 200, 400]:
    created = now - timedelta(days=days)
    print(f"age {days:>3}d -> {p.action(created, now)}")


A background worker (think cron + queue) iterates through the
metadata store, asks the policy what to do, and performs the action.
This is one of those places where **separating metadata from data**
pays off again — we don't need to scan petabytes to decide what to
move, we just walk a database table.



## 🧭 Closing thoughts

The simple theme across all three notebooks:

- **Small metadata plane, enormous data plane**, each with the tech that fits it.
  The data plane is the easy half: immutable blobs, written once, read
  sequentially. The index over them is where the design lives.
- **Erasure coding** for cheap durability — and name its costs (k-way read
  amplification on repair, tail latency across k nodes, hopeless for small
  objects) or you have only learned half of it.
- **Durability is a function of repair time**, not of disk quality. Halving MTTR
  bought more nines in Notebook 1 than adding a parity shard did, and cost nothing.
  The advertised 11 nines is a floor set by correlated failure and operator
  error, not the output of the binomial.
- **Multipart upload** for big objects; **presigned URLs** for safe sharing;
  **consistent hashing** for smooth growth; **range-partitioned metadata** so
  `LIST` stays affordable; **lifecycle policies** for cost control (including
  aborting incomplete uploads — the invisible line on the bill).
- Every feature in a real S3 clone comes from a problem you can see break in
  ~50 lines of Python. Try modifying the code above — e.g. make the ring use
  only 1 vnode per node and watch the load imbalance, or shorten the erasure
  code's repair window and watch the nines move.